<a href="https://colab.research.google.com/github/boymanews/Ghost/blob/main/Cover%20Music%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 align="center">🎵 YuE 2.0 AI Music Cover Studio</h1>
<p align="center">
  <img src="https://img.shields.io/badge/Model-YuE2--3B--FP16-ff69b4?style=for-the-badge&logo=huggingface" />
  <img src="https://img.shields.io/badge/Transcriber-SheetSage2-purple?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Backend-ComfyUI%20Native-blue?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Hardware-T4%20%7C%20A100%20%7C%20L4%20GPU-green?style=for-the-badge&logo=nvidia" />
</p>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

---
### 💡 How AI Music Covers Work
1. **Input**: You provide an audio track (original song) and its lyrics.
2. **Transcription**: **SheetSage2** listens to the reference song and transcribes its vocal melody and musical timing into symbolic score.
3. **Re-Imagining**: **YuE 2.0** composes a brand new cover performance matching that exact melody, transformed by your target genre, vocal character, and gender choice.

In [1]:
# @title 1. Hardware Verification & GPU Diagnostics
# @markdown Checks CUDA environment and cleans system memory.
import gc
import torch
from IPython.display import HTML, display

if not torch.cuda.is_available():
    display(HTML("<div style='background-color:#ffebee;padding:12px;border-radius:8px;border-left:5px solid #f44336;'><h3 style='color:#c62828;margin:0;'>⚠️ GPU Not Detected!</h3><p style='margin:5px 0 0 0;'>Please navigate to <b>Runtime &rarr; Change runtime type</b> and select <b>T4 GPU</b> or higher.</p></div>"))
    raise SystemExit("GPU accelerator missing.")

gc.collect()
torch.cuda.empty_cache()

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

GPU Device:,Tesla T4
Total VRAM:,14.56 GB
CUDA Version:,13.0
PyTorch:,2.11.0+cu130


In [2]:
# @title 2. Install Engine Dependencies
# @markdown Clones ComfyUI and installs all required audio processing libraries.
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI audio cover environment...")

if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors huggingface_hub librosa
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/audio_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/input", exist_ok=True)
os.makedirs("/content/ComfyUI/output/audio", exist_ok=True)

display(HTML("<div style='background-color:#e3f2fd;padding:12px;border-radius:8px;border-left:5px solid #2196f3;'><b style='color:#1565c0;'>✅ Environment installed and ready!</b></div>"))

🚀 Setting up ComfyUI audio cover environment...
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 56617, done.
remote: Counting objects: 100% (1656/1656), done.
remote: Compressing objects: 100% (511/511), done.
remote: Total 56617 (delta 1438), reused 1176 (delta 1143), pack-reused 54961 (from 3)
Receiving objects: 100% (56617/56617), 93.88 MiB | 15.22 MiB/s, done.
Resolving deltas: 100% (38625/38625), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.7 MB/s eta 0:00:00
   

In [ ]:
# @title 3. Download Models & Launch Cover Backend
# @markdown Downloads `yue2_3b_fp16.safetensors` & `sheetsage2_bf16.safetensors`, then starts the engine.
import os
import time
import subprocess
import urllib.request
from huggingface_hub import hf_hub_download
from IPython.display import HTML, display

%cd /content/ComfyUI

ckpt_dir = "/content/ComfyUI/models/checkpoints"
encoder_dir = "/content/ComfyUI/models/audio_encoders"
ckpt_path = os.path.join(ckpt_dir, "yue2_3b_fp16.safetensors")
encoder_path = os.path.join(encoder_dir, "sheetsage2_bf16.safetensors")

# 1. Download YuE2 Checkpoint
if not os.path.exists(ckpt_path) or os.path.getsize(ckpt_path) < 6_000_000_000:
    print("📥 Downloading YuE2 FP16 checkpoint (~7.26 GB)... Please wait.")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Jokality/YuE2-fp16/resolve/main/checkpoints/yue2_3b_fp16.safetensors" \
      -d "{ckpt_dir}" \
      -o "yue2_3b_fp16.safetensors"
else:
    print("⚡ Found existing YuE2 FP16 checkpoint in cache.")

# 2. Download SheetSage2 Audio Encoder
if not os.path.exists(encoder_path) or os.path.getsize(encoder_path) < 1_000_000_000:
    print("📥 Downloading SheetSage2 BF16 Audio Encoder (~1.39 GB)... Please wait.")
    hf_hub_download(
        repo_id="Comfy-Org/YuE2",
        filename="audio_encoders/sheetsage2_bf16.safetensors",
        local_dir="/content/ComfyUI/models"
    )
else:
    print("⚡ Found existing SheetSage2 audio encoder in cache.")

# Memory safeguards for Colab T4 host RAM
os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clean up stale processes
!pkill -9 -f "main.py" || true
time.sleep(2)

print("🔄 Starting ComfyUI Cover Engine with RAM Management...")
server_log = open("/content/comfy_server.log", "w")

server_cmd = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--force-fp16",
    "--reserve-vram", "1.0",
    "--dont-upcast-attention"
]

proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/content/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("""
    <div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
      <b style='color:#2e7d32;'>🟢 YuE2 Music Cover Engine Online!</b><br>
      <span style='color:#555;'>Models loaded &bull; Fast compute active. Run <b>Cell 4</b>.</span>
    </div>
    """))
else:
    print("\n❌ Startup failed. Log contents (/content/comfy_server.log):\n" + "-"*60)
    with open("/content/comfy_server.log", "r") as f:
        print(f.read())
    print("-" * 60)

/content/ComfyUI
📥 Downloading YuE2 FP16 checkpoint (~7.26 GB)... Please wait.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
7c0576|OK  |   150MiB/s|/content/ComfyUI/models/checkpoints/yue2_3b_fp16.safetensors

Status Legend:
(OK):download completed.
📥 Downloading SheetSage2 BF16 Audio Encoder (~1.39 GB)... Please wait.


audio_encoders/sheetsage2_bf16.safetenso(…): reconstructing file:   0%|          |  0.00B / 1.39GB            

audio_encoders/sheetsage2_bf16.safetenso(…): downloading bytes:           |  0.00B            

^C


In [ ]:
# @title 4. Interactive Music Cover Studio
# @markdown Configure your cover genre, vocal transformation, and upload your reference song below.
import os
import gc
import json
import time
import glob
import uuid
import random
import shutil
import subprocess
import urllib.request
import torch
import websocket
from google.colab import files
from IPython.display import HTML, display, clear_output

# =========================================================================
# 🎚️ 1. AUDIO INPUT
# =========================================================================
# @markdown ### 🎧 1. Reference Track
# @markdown *Uncheck `upload_new_audio` if you want to reuse the last uploaded track.*
upload_new_audio = True #@param {type:"boolean"}
use_existing_file = "reference_song.mp3" #@param {type:"string"}

# =========================================================================
# 🎚️ 2. COVER GENRE & VOCAL CHARACTER
# =========================================================================
# @markdown ### 🎼 2. Cover Transformation Recipe
cover_genre_preset = "Custom / Manual Prompt" #@param ["Custom / Manual Prompt", "Acoustic Folk Unplugged", "Female-to-Deep Male Rock", "Male-to-Female Pop", "80s Vintage Synthwave Flip", "Intimate Late-Night Jazz Lounge", "Aggressive Modern Metalcore", "Lofi Chillhop Coffee Shop", "Epic Orchestral Trailer"]

# @markdown **Vocal Character & Gender Override:** Automatically dials in the singer's vocal cords, range, and octave placement.
vocal_character = "Deep Male Baritone (Singing 1 Octave Down)" #@param ["Keep Same as Reference / Neutral", "Deep Male Baritone (Singing 1 Octave Down)", "Raw Gritty Alt-Rock Male", "High Soaring Male Rock Tenor", "Warm Acoustic Female Alto", "Airy Dream-Pop Female Soprano", "Smooth Soul / R&B Crooner", "Vintage 1950s Jazz Male Crooner"]

custom_style_override = "Intimate acoustic cover, clean fingerstyle acoustic guitar, warm upright bass, gentle shaker, natural studio room reverb, raw unpolished emotional delivery" #@param {type:"string"}

# =========================================================================
# 🎚️ 3. MELODY FIDELITY & COVER SCOPE
# =========================================================================
# @markdown ### 🎙️ 3. Melody Tracking & Style Strength
# @markdown - **Melody Only (Recommended)**: Keeps the original vocal melody, completely replaces the backing instrumentation.
# @markdown - **Full Band**: Re-harmonizes both the vocal melody and original chord progression.
cover_scope = "Melody Only (Vocal Line - Recommended)" #@param ["Melody Only (Vocal Line - Recommended)", "Full Band (Melody + Harmony Chords)"]

# @markdown **Melody Adherence:** How strictly the new singer must stick to original pitch.
melody_lock = "Balanced (Authentic Cover)" #@param ["Strict (Exact Pitch & Timing)", "Balanced (Authentic Cover)", "Creative (Allows Expressive Ad-libbing)"]

# @markdown **Genre Strength (CFG Scale):** How strongly to push the new target genre.
genre_strength = 1.2 #@param {type:"slider", min:1.0, max:2.2, step:0.1}

# =========================================================================
# 🎚️ 4. LYRICS
# =========================================================================
# @markdown ### 📝 4. Song Lyrics
# @markdown *Enter the lyrics for your song. Syllables should align with the reference rhythm.*
lyrics = "[Verse]\nStreetlights shimmer on the pavement\nMidnight breathing soft and slow\nEvery corner holds a memory\nOf the places we used to know\n\n[Chorus]\nStay with me until the morning\nLet the city fade away\nWe don't need to chase tomorrow\nAll I need is here today\n\n[Verse]\nOld caf\u00e9 beneath the neon\nRain is falling on the glass\nEvery song becomes a story\nEvery moment moves too fast\n\n[Chorus]\nStay with me until the morning\nLet the city fade away\nWe don't need to chase tomorrow\nAll I need is here today\n\n[Bridge]\nMaybe time will keep on turning\nMaybe everything will change\nBut tonight the music holds us\nLike a fire inside the rain\n\n[Chorus]\nStay with me until the morning\nLet the city fade away\nWe don't need to chase tomorrow\nAll I need is here today" #@param {type:"string"}

# =========================================================================
# 🎚️ 5. EXPORT & QUALITY
# =========================================================================
# @markdown ### ⚙️ 5. Quality & File Format
generation_mode = "Balanced Quality (DPM_2 - 32 steps)" #@param ["Fast Draft (Euler - 22 steps)", "Balanced Quality (DPM_2 - 32 steps)", "Ultra High Fidelity (DPM_2 - 38 steps)"]
seed = 0 #@param {type:"integer"}
randomize_seed = True #@param {type:"boolean"}
audio_format = "wav" #@param ["mp3", "flac", "wav"]
filename_prefix = "audio/YuE2_Cover" #@param {type:"string"}

# Safety ceiling (song duration is automatically controlled by reference track)
MAX_DURATION_SECONDS = 360

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================================================================
# 🛡️ DAEMON HEALTH VERIFICATION
# =========================================================================
def ensure_comfy_server():
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1.5)
        return True
    except Exception:
        pass

    print("⚠️ Engine backend was offline. Launching daemon now...")
    os.system("pkill -9 -f 'main.py' || true")
    time.sleep(1)

    server_log = open("/content/comfy_server.log", "w")
    server_cmd = [
        "python", "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--force-fp16",
        "--reserve-vram", "1.0",
        "--dont-upcast-attention"
    ]
    subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/content/ComfyUI")

    for _ in range(40):
        try:
            urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
            print("🟢 Cover Engine is ready!")
            return True
        except Exception:
            time.sleep(1.5)
    return False

if not ensure_comfy_server():
    raise RuntimeError("Could not connect to ComfyUI backend. Check /content/comfy_server.log")

# =========================================================================
# 📥 REFERENCE TRACK INGESTION
# =========================================================================
input_dir = "/content/ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)

if upload_new_audio:
    print("🎵 Select your reference audio file from your device (MP3, WAV, FLAC, M4A)...\n")
    uploaded_dict = files.upload()
    if not uploaded_dict:
        raise ValueError("❌ No file selected. Please re-run and select your track.")
    clear_output(wait=True)
    raw_name = list(uploaded_dict.keys())[0]
    clean_filename = raw_name.replace(" ", "_").replace("(", "_").replace(")", "_")
    target_path = os.path.join(input_dir, clean_filename)
    with open(target_path, "wb") as f:
        f.write(uploaded_dict[raw_name])
    reference_filename = clean_filename
    del uploaded_dict
    gc.collect()
else:
    reference_filename = use_existing_file
    target_path = os.path.join(input_dir, reference_filename)
    if not os.path.exists(target_path):
        # Search for any audio in input
        any_audio = glob.glob(f"{input_dir}/*.*")
        if any_audio:
            reference_filename = os.path.basename(any_audio[0])
            target_path = any_audio[0]
        else:
            raise FileNotFoundError(f"❌ Audio file '{reference_filename}' not found in ComfyUI/input/. Please check 'upload_new_audio'.")

file_size_mb = round(os.path.getsize(target_path) / (1024 * 1024), 2)
print(f"✅ Reference song ready: {reference_filename} ({file_size_mb} MB)")

# =========================================================================
# 🧩 STYLE PROMPT & PARAMETER COMPOSITION
# =========================================================================
genre_recipes = {
    "Acoustic Folk Unplugged": "Acoustic folk cover, warm fingerstyle acoustic guitars, gentle upright bass, subtle shaker percussion, natural live recording, intimate studio acoustic atmosphere",
    "Female-to-Deep Male Rock": "Alternative grunge rock cover, driving electric guitars, crunchy overdriven bassline, live acoustic drum kit, anthemic 90s rock production",
    "Male-to-Female Pop": "Modern indie-pop cover, bright chiming guitars, lush atmospheric synth pads, punchy electronic drums, melodic bassline, polished commercial summer production",
    "80s Vintage Synthwave Flip": "1980s retro synthwave cover, analog vintage synthesizers, gated snare reverb, driving arpeggiated bassline, neon nocturnal aesthetic, cinematic night drive",
    "Intimate Late-Night Jazz Lounge": "Late-night jazz club cover, Rhodes electric piano, upright acoustic bass, brushed snare drums, warm tenor saxophone accents, relaxed 85 BPM lounge atmosphere",
    "Aggressive Modern Metalcore": "Heavy metalcore cover, down-tuned drop-D distorted guitars, thunderous double bass drum kit, aggressive driving energy, explosive stadium chorus",
    "Lofi Chillhop Coffee Shop": "Warm lofi hip hop cover, gentle vinyl crackle, mellow electric piano chords, relaxed boom-bap drums, deep warm sub bass, peaceful afternoon coffee shop atmosphere",
    "Epic Orchestral Trailer": "Epic cinematic orchestral trailer cover, soaring brass section, expressive emotional violins and cellos, thundering cinematic percussion, dramatic heroic crescendo",
    "Custom / Manual Prompt": custom_style_override
}

vocal_recipes = {
    "Keep Same as Reference / Neutral": "expressive lead vocals",
    "Deep Male Baritone (Singing 1 Octave Down)": "deep resonant male baritone vocal, singing one octave lower, rich chest resonance, grounded warm vocal delivery",
    "Raw Gritty Alt-Rock Male": "raw gritty male baritone lead vocal, vocal fry on phrase ends, raspy emotive rock delivery, powerful chest voice",
    "High Soaring Male Rock Tenor": "high soaring male rock tenor, powerful chest-belt in high register, passionate melodic rock delivery",
    "Warm Acoustic Female Alto": "warm expressive female alto lead vocal, delicate vibrato, intimate breathy close-mic recording",
    "Airy Dream-Pop Female Soprano": "airy female soprano vocal, ethereal head-voice, bright crystal clear vocal tone, modern reverb sheen",
    "Smooth Soul / R&B Crooner": "smooth expressive male R&B vocal, silky falsetto runs, effortless soulful phrasing, warm melodic tone",
    "Vintage 1950s Jazz Male Crooner": "classic mid-century male jazz crooner, velvety smooth baritone, classic phrasing, vintage tube microphone warmth"
}

# Build merged style prompt
base_genre = custom_style_override if cover_genre_preset == "Custom / Manual Prompt" else genre_recipes[cover_genre_preset]
vocal_tag = vocal_recipes[vocal_character]
active_style = f"{vocal_tag}, {base_genre}"

# Map parameters
mode_code = "melody" if "Melody Only" in cover_scope else "full"
temperature_map = {
    "Strict (Exact Pitch & Timing)": 0.75,
    "Balanced (Authentic Cover)": 0.85,
    "Creative (Allows Expressive Ad-libbing)": 0.95
}
active_temp = temperature_map[melody_lock]

sampler_map = {
    "Fast Draft (Euler - 22 steps)": ("euler", "sgm_uniform", 22),
    "Balanced Quality (DPM_2 - 32 steps)": ("dpm_2", "sgm_uniform", 32),
    "Ultra High Fidelity (DPM_2 - 38 steps)": ("dpm_2", "sgm_uniform", 38)
}
sampler_name, scheduler, steps = sampler_map[generation_mode]

active_seed = random.randint(1, 2147483647) if randomize_seed else seed
ckpt_name = "yue2_3b_fp16.safetensors"
encoder_name = "sheetsage2_bf16.safetensors"

# Clear memory before starting prompt
try:
    free_req = urllib.request.Request(
        "http://127.0.0.1:8188/free",
        data=b'{"unload_models": false, "free_memory": true}',
        headers={"Content-Type": "application/json"}
    )
    urllib.request.urlopen(free_req, timeout=1)
except Exception:
    pass

# =========================================================================
# 🔄 WORKFLOW DEFINITION
# =========================================================================
workflow_api = {
    "10": {
        "inputs": {
            "filename_prefix": filename_prefix,
            "format": audio_format,
            "format.quality": "320k",
            "audio": ["33:9", 0]
        },
        "class_type": "SaveAudioAdvanced"
    },
    "45": {
        "inputs": {
            "audio": reference_filename
        },
        "class_type": "LoadAudio"
    },
    "33:15": {
        "inputs": {
            "ckpt_name": ckpt_name
        },
        "class_type": "CheckpointLoaderSimple"
    },
    "33:43": {
        "inputs": {
            "audio_encoder_name": encoder_name
        },
        "class_type": "AudioEncoderLoader"
    },
    "33:41": {
        "inputs": {
            "mode": mode_code,
            "audio_encoder": ["33:43", 0],
            "audio": ["45", 0]
        },
        "class_type": "SheetSage2AudioToABC"
    },
    "33:44": {
        "inputs": {
            "source": ["33:41", 0]
        },
        "class_type": "PreviewAny"
    },
    "33:25": {
        "inputs": {
            "style": active_style,
            "lyrics": lyrics,
            "abc": ["33:44", 0],
            "seed": ["33:34", 0],
            "mode": mode_code,
            "max_duration": MAX_DURATION_SECONDS,
            "temperature": active_temp,
            "top_p": 0.95,
            "top_k": 100,
            "repetition_penalty": 1.2,
            "cfg_scale": genre_strength,
            "clip": ["33:15", 1]
        },
        "class_type": "YuE2GenerateMusic"
    },
    "33:18": {
        "inputs": {
            "conditioning": ["33:25", 0]
        },
        "class_type": "ConditioningZeroOut"
    },
    "33:8": {
        "inputs": {
            "seed": ["33:34", 0],
            "steps": steps,
            "cfg": 1.0,
            "sampler_name": sampler_name,
            "scheduler": scheduler,
            "denoise": 1.0,
            "model": ["33:15", 0],
            "positive": ["33:25", 0],
            "negative": ["33:18", 0],
            "latent_image": ["33:5", 0]
        },
        "class_type": "KSampler"
    },
    "33:5": {
        "inputs": {
            "seconds": ["33:25", 1],
            "batch_size": 1
        },
        "class_type": "EmptyYuE2LatentAudio"
    },
    "33:9": {
        "inputs": {
            "samples": ["33:8", 0],
            "vae": ["33:15", 2]
        },
        "class_type": "VAEDecodeAudio"
    },
    "33:34": {
        "inputs": {
            "seed": active_seed
        },
        "class_type": "SeedNode"
    }
}

node_names = {
    "45": f"LoadAudio ({reference_filename})",
    "33:43": "AudioEncoderLoader (SheetSage2)",
    "33:41": f"SheetSage2AudioToABC ({mode_code.upper()} transcription)",
    "33:15": "CheckpointLoaderSimple (YuE2 FP16)",
    "33:25": f"YuE2GenerateMusic (Acoustic Conditioning - {vocal_character})",
    "33:5": "EmptyYuE2LatentAudio (Dynamic Latent Allocation)",
    "33:8": f"KSampler ({sampler_name}/{scheduler} - {steps} steps)",
    "33:9": "VAEDecodeAudio (Audio Reconstruction)",
    "10": "SaveAudioAdvanced (Final Export)"
}

# Display user-friendly job card
display(HTML(f"""
<div style='background:#18181b;color:#f4f4f5;border:1px solid #3f3f46;border-radius:12px;padding:18px;margin-bottom:15px;font-family:sans-serif;'>
  <div style='display:flex;align-items:center;justify-content:space-between;border-bottom:1px solid #27272a;padding-bottom:10px;margin-bottom:12px;'>
    <h3 style='margin:0;color:#ec4899;font-size:16px;'>🎙️ YuE 2.0 AI Cover Synthesis Initiated</h3>
    <span style='background:#10b98120;color:#34d399;padding:3px 10px;border-radius:12px;font-size:11px;font-weight:bold;'>DISPATCHED</span>
  </div>
  <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:10px;font-size:12px;'>
    <div><b>Reference Audio:</b> <span style='color:#38bdf8;'>{reference_filename}</span></div>
    <div><b>Cover Profile:</b> <span style='color:#a78bfa;'>{cover_genre_preset}</span></div>
    <div><b>Vocal Character:</b> <span style='color:#f472b6;'>{vocal_character}</span></div>
    <div><b>Tracking Scope:</b> <span style='color:#facc15;'>{cover_scope}</span></div>
    <div><b>Melody Fidelity:</b> <span style='color:#4ade80;'>{melody_lock}</span></div>
    <div><b>Seed:</b> <span style='color:#94a3b8;'>{active_seed}</span></div>
  </div>
</div>
"""))

client_id = str(uuid.uuid4())
ws = websocket.WebSocket()
ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")

req_payload = json.dumps({"prompt": workflow_api, "client_id": client_id}).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_payload, headers={'Content-Type': 'application/json'})
resp = urllib.request.urlopen(req)
prompt_id = json.loads(resp.read())['prompt_id']

print(f"📡 Tracking Task ID: {prompt_id} | Active Seed: {active_seed}")
print("=" * 68)

start_time = time.time()
job_failed = False

while True:
    msg = ws.recv()
    if isinstance(msg, str):
        event = json.loads(msg)
        event_type = event.get('type')
        event_data = event.get('data', {})

        if event_type == 'execution_error':
            print(f"\n❌ Pipeline Error in Node {event_data.get('node_id')}: {event_data.get('exception_message')}")
            job_failed = True
            break

        if event_type == 'executing':
            node_id = event_data.get('node')
            if node_id is None:
                if event_data.get('prompt_id') == prompt_id:
                    print("\n\n🎉 Cover Synthesis Completed Successfully!")
                    break
            else:
                stage_label = node_names.get(str(node_id), f"Node {node_id}")
                print(f"\n▶️ [Stage: {stage_label}]")

                # Evict SheetSage2 from CPU memory right after ABC extraction to safeguard RAM
                if str(node_id) == "33:44":
                    try:
                        purge_req = urllib.request.Request(
                            "http://127.0.0.1:8188/free",
                            data=b'{"unload_models": true, "free_memory": true}',
                            headers={"Content-Type": "application/json"}
                        )
                        urllib.request.urlopen(purge_req, timeout=1)
                        print("🧹 Freed SheetSage2 transcription model from host RAM.")
                    except Exception:
                        pass

                # Flush lingering attention caches before KSampler diffusion
                if str(node_id) == "33:8":
                    try:
                        flush_req = urllib.request.Request(
                            "http://127.0.0.1:8188/free",
                            data=b'{"unload_models": false, "free_memory": true}',
                            headers={"Content-Type": "application/json"}
                        )
                        urllib.request.urlopen(flush_req, timeout=1)
                    except Exception:
                        pass

        elif event_type == 'progress':
            curr_val = event_data.get('value', 0)
            max_val = event_data.get('max', steps)
            pct = int((curr_val / max_val) * 100)
            bar_len = 25
            filled = int(bar_len * curr_val // max_val)
            bar = '█' * filled + '░' * (bar_len - filled)
            elapsed_s = int(time.time() - start_time)
            print(f"\r   Progress: [{bar}] {curr_val}/{max_val} ({pct}%) | {elapsed_s}s elapsed", end="", flush=True)

ws.close()
total_seconds = round(time.time() - start_time, 1)

# Display audio output card
if not job_failed:
    latest_file = None
    try:
        hist_req = urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}")
        hist_data = json.loads(hist_req.read())
        outputs = hist_data.get(prompt_id, {}).get("outputs", {})
        for node_out in outputs.values():
            if "audio" in node_out:
                filename = node_out["audio"][0]["filename"]
                subfolder = node_out["audio"][0].get("subfolder", "")
                latest_file = os.path.join("/content/ComfyUI/output", subfolder, filename)
                break
    except Exception:
        pass

    if not latest_file or not os.path.exists(latest_file):
        candidate_files = glob.glob(f"/content/ComfyUI/output/{filename_prefix}*.{audio_format}")
        if not candidate_files:
            candidate_files = glob.glob(f"/content/ComfyUI/output/audio/*.{audio_format}")
            if not candidate_files:
                candidate_files = glob.glob("/content/ComfyUI/output/*.*")
        if candidate_files:
            latest_file = max(candidate_files, key=os.path.getctime)

    if latest_file and os.path.exists(latest_file):
        file_size_mb = round(os.path.getsize(latest_file) / (1024 * 1024), 2)
        filename_base = os.path.basename(latest_file)

        try:
            _d = float(subprocess.check_output([
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=nw=1:nk=1",
                latest_file
            ]).strip())
            actual_duration_txt = f"{int(_d//60)}:{int(_d%60):02d}"
        except Exception:
            actual_duration_txt = "n/a"

        import base64
        with open(latest_file, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")

        mime_type = "audio/mpeg" if audio_format == "mp3" else f"audio/{audio_format}"

        display(HTML(f"""
        <div style='background:#121214;border:1px solid #27272a;border-radius:14px;padding:22px;max-width:680px;font-family:sans-serif;color:#f4f4f5;box-shadow:0 10px 25px rgba(0,0,0,0.5);margin-top:15px;'>
          <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:14px;'>
            <div style='display:flex;align-items:center;gap:10px;'>
              <div style='background:linear-gradient(135deg,#ec4899,#8b5cf6);width:42px;height:42px;border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:22px;'>🎙️</div>
              <div>
                <h3 style='margin:0;font-size:16px;color:#fff;'>{filename_base}</h3>
                <span style='color:#a1a1aa;font-size:12px;'>Cover Rendered in <b>{total_seconds}s</b> &bull; {file_size_mb} MB &bull; {actual_duration_txt} Audio</span>
              </div>
            </div>
            <span style='background:#22c55e20;color:#4ade80;padding:4px 10px;border-radius:20px;font-size:11px;font-weight:bold;'>COVER READY</span>
          </div>

          <audio controls autoplay style='width:100%;margin:12px 0;border-radius:8px;'>
            <source src='data:{mime_type};base64,{audio_b64}' type='{mime_type}'>
          </audio>

          <div style='background:#18181b;padding:12px;border-radius:8px;font-size:12px;margin-top:10px;border:1px solid #27272a;'>
            <div><b>Reference Song:</b> <span style='color:#38bdf8;'>{reference_filename}</span></div>
            <div style='margin-top:4px;'><b>Vocal & Genre Style:</b> <span style='color:#cbd5e1;'>{active_style[:140]}...</span></div>
            <div style='display:flex;justify-content:space-between;margin-top:8px;'>
              <span><b>Vocal Character:</b> <code style='color:#f472b6;'>{vocal_character}</code></span>
              <span><b>Mode:</b> {mode_code.upper()}</span>
              <span><b>Seed:</b> {active_seed}</span>
            </div>
          </div>
        </div>
        """))
    else:
        print("⚠️ No cover audio output found. Check /content/comfy_server.log for details.")

In [ ]:
# @title 5. File Manager: Download All Generated Covers
# @markdown Compresses all audio files in the output directory into a single `.zip` and prompts a browser download.
import glob
import os
import shutil
from google.colab import files

output_dir = "/content/ComfyUI/output/audio"
all_audio = glob.glob(f"{output_dir}/*.*")

if not all_audio:
    print("No audio tracks generated yet.")
else:
    print(f"Found {len(all_audio)} track(s):")
    for track in all_audio:
        size_mb = round(os.path.getsize(track) / (1024*1024), 2)
        print(f" - {os.path.basename(track)} ({size_mb} MB)")

    zip_name = "/content/YuE2_Generated_Covers"
    shutil.make_archive(zip_name, 'zip', output_dir)
    print("\n📦 Downloading cover tracks zip to your local computer...")
    files.download(f"{zip_name}.zip")